# Reading Raw Data

In [0]:
df = spark.table("workspace.raw_crashes.people_table")

In [0]:
df.display()

# Data Transformation

## Fix Date datatype

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

df = df.withColumn("CRASH_DATE", substring_index("CRASH_DATE", " ", 1))
df = df.withColumn("CRASH_DATE", to_date(col("CRASH_DATE"), "MM/dd/yyyy"))

# df.withColumnRenamed('CRASH_RECORD_ID', 'id').display()

## Trimming

In [0]:
#trim the string columns
# normalization 

for field in df.schema.fields:
    print(field)
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

# Cast AGE column to integer, replacing nulls with 0
from pyspark.sql.types import IntegerType

df = df.withColumn("AGE", expr("try_cast(AGE as int)"))
df.display()

In [0]:
df = df.fillna({'AGE': 0})

df.display()

In [0]:
from pyspark.sql import types

def process_age(x):
    if int(x):
        return int(x)
    return 0
    

processing_udf =udf(process_age, types.IntegerType())

df =df.withColumn('AGE', processing_udf(col('AGE')))

In [0]:
from pyspark.sql.functions import sum 

cols_to_keep = []
df_nulls = df.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
]).collect()[0].asDict()

print(df_nulls)


for key, value in df_nulls.items():
    if value < 50000:
        cols_to_keep.append(key)
        


df = df.select(cols_to_keep)




# Remove & Cleaning Nulls

In [0]:
string_nulls = ["", " ", "NA", "N/A", "null", "None"]
numeric_nulls = [0, -1, 9999]
df_clean =df

df.dtypes

for column, data_type in df.dtypes:
    if data_type == 'string':
        df_clean = df_clean.fillna({column: 'UNKNOWN'}) 
     
        
    elif data_type in ("int", "bigint", "double", "float"):
        df_clean = df_clean.fillna({column: 0}) 

         
df_clean = df_clean.withColumn("AGE", when(col("AGE") < 0, 0)
                                            .otherwise(col("AGE")))
 



# WRITE TO CLEAN SILVER TABLE

In [0]:



df_clean.write \
    .mode('overwrite') \
    .format("delta") \
    .saveAsTable("workspace.silver_crashes.person_table")
